##### Data processing 
there are a number of things we need to filter for in this step 
- all species are present 
- length 
- correctly masked exonic sequences 

In [1]:
import numpy as np

from cogent3.app import get_app
from cogent3.app.io import open_data_store
from cogent3.app.composable import define_app
from cogent3.app.typing import AlignedSeqsType
from cogent3.core.alignment import make_aligned_seqs
from remove_mask import has_mask_run, collapse_mask_runs

/var/folders/1p/g1mgr9l91fggf5jlnsqpcf1c6sw6jz/T/ipykernel_49189/2507624614.py:5: DeprecationWarning: cogent3.app.composable is deprecated and will be removed in 2026.9. Import from scinexus.composable instead.
  from cogent3.app.composable import define_app


In [2]:
@define_app
def remove_mask(
    aln: AlignedSeqsType,
    ref_name="dmel",
    mask_value=16,
    detect_run_length=40,
    collapse_run_length=1,
) -> AlignedSeqsType:
    """Detect long masked exon in reference and collapse runs."""

    seqs = aln.to_dict(as_array=True)
    reference = seqs[ref_name]

    if not has_mask_run(reference, mask_value, detect_run_length):
        raise ValueError(f"{aln.source} contains no masked exon")

    collapsed = collapse_mask_runs(
        reference,
        seqs,
        mask_value=mask_value,
        min_run_length=collapse_run_length,
    )

    return make_aligned_seqs(
        collapsed,
        moltype=aln.moltype,
        source=aln.source,
        ) # type: ignore

In [3]:
def renamer(old_name):
    if "melanogaster" in old_name:
        return "dmel"
    if "simulans" in old_name:
        return "dsim"
    if "yakuba" in old_name:
        return "dyak"
    return old_name


@define_app
def rename_seqs(aln: AlignedSeqsType) -> AlignedSeqsType:
    return aln.renamed_seqs(renamer)

In [4]:
# dstore set up 
align_dir = open_data_store("~/repos/mdeq-cpg/data/raw", suffix="fa")
out_dir = open_data_store("~/repos/mdeq-cpg/data/processed", suffix="fa", mode="w")

In [5]:
# apps 
loader = get_app("load_aligned", moltype="dna")
rename_seqs_app = rename_seqs()
take_seqs_app = get_app("take_named_seqs", "dmel", "dsim", "dyak")
mask_check_app = remove_mask()
min_len_app = get_app("min_length", 300)
omit_gaps_app = get_app("omit_gap_pos", motif_length=2, allowed_frac=0.0)
writer = get_app("write_seqs", out_dir)

# process
proc = loader + rename_seqs_app + take_seqs_app + mask_check_app + min_len_app + omit_gaps_app + writer

In [6]:
r = proc.apply_to(align_dir, show_progress=True)

           0/27092 [00:00<?]

In [7]:
out_dir.summary_not_completed

type,origin,message,num,source
FAIL,min_length,'202 < min_length 300... 62 < min_length 300',6095,"FBgn0036128-0, FBgn0025457-1, FBgn0031317-0, ..."
ERROR,remove_mask,'ValueError: FBgn0039...tains no masked exon',19766,"FBgn0031044-1, FBgn0036781-1, FBgn0033079-1, ..."
FAIL,take_named_seqs,"""named seq(s) {'dsim'... in ('dyak', 'dmel')""",602,"FBgn0036082-0, FBgn0260869-0, FBgn0267398, ..."
ERROR,rename_seqs,'ValueError: non-uniq...amer at 0x11840b560>',107,"FBgn0053884, FBgn0053826, FBgn0053877, ..."


In [8]:
r.describe

Condition,Value
completed,522
not_completed,26570
logs,1


In [9]:
# take a look at a random alignment from the filtered alignment directory 

i = np.random.choice(500)
aln = loader(out_dir[i])

print("length of alignment: ", len(aln))

fig = aln.dotplot()
fig.show()

length of alignment:  434
